# 🔧 MetaboNet Clean Core Reconstruction v7

**Goal**: Build clean, high-density dataset with core diabetes features only  
**Method**: Polars Lazy API for memory efficiency  
**Output**: `master_train_v7_clean_core.parquet`

In [ ]:
!pip install polars -U -q

import polars as pl
import numpy as np
import re
import os
from datetime import datetime

# Google Colab mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False

print(f"✅ Polars: {pl.__version__}")
print(f"📍 Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
if IN_COLAB:
    BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
else:
    BASE_PATH = r"D:\Diabete\data"

RAW_TRAIN = f"{BASE_PATH}/raw/metabonet_public_train.parquet"
OUTPUT_FILE = f"{BASE_PATH}/processed/master_train_v7_clean_core.parquet"

os.makedirs(f"{BASE_PATH}/processed", exist_ok=True)

# STRICT CORE COLUMNS ONLY
CORE_COLUMNS = [
    'id', 'date',
    'CGM',
    'basal', 'bolus', 'insulin_type_bolus',
    'carbs',
    'age', 'weight', 'gender'
]

# EXPLICITLY EXCLUDED (for documentation)
EXCLUDED = [
    'galvanic_skin_response', 'skin_temp', 'air_temp', 'heartrate', 'steps',
    'calories_burned', 'workout_duration', 'workout_intensity', 'workout_label',
    'activity', 'is_test', 'is_pregnant', 'age_of_diagnosis', 'source_file', 
    'misc_notes', 'randomization_date', 'cgm_device', 'insulin_delivery_device'
]

print(f"📁 Input:  {RAW_TRAIN}")
print(f"📁 Output: {OUTPUT_FILE}")
print(f"\n🎯 Core Columns: {CORE_COLUMNS}")

## Step 1: The Universal ASCII Decoder

In [ ]:
def decode_ascii(val: str) -> str:
    """
    Decode stringified ASCII byte arrays to readable text.
    Example: "{77, 97, 108, 101}" -> "Male"
    
    Handles:
    - "{77, 97, 108, 101}" format
    - "[77, 97, 108, 101]" format  
    - Already decoded strings (pass-through)
    """
    if val is None:
        return "Unknown"
    
    val = str(val).strip()
    
    # Check if it's an ASCII byte array format
    if val.startswith('{') and val.endswith('}'):
        inner = val[1:-1]  # Remove braces
    elif val.startswith('[') and val.endswith(']'):
        inner = val[1:-1]  # Remove brackets
    else:
        # Already a plain string, return as-is
        return val if val else "Unknown"
    
    try:
        # Split by comma, convert each to int, then to char
        codes = [int(x.strip()) for x in inner.split(',') if x.strip()]
        decoded = ''.join(chr(c) for c in codes)
        return decoded if decoded else "Unknown"
    except (ValueError, TypeError):
        return "Unknown"


# Test the decoder
print("🧪 Testing decoder:")
test_cases = [
    "{77, 97, 108, 101}",      # "Male"
    "{70, 101, 109, 97, 108, 101}",  # "Female"
    "{114, 97, 112, 105, 100}",  # "rapid"
    "AlreadyDecoded",
    None,
    ""
]

for tc in test_cases:
    print(f"   '{tc}' -> '{decode_ascii(tc)}'")

In [ ]:
def apply_decoder_to_column(df: pl.LazyFrame, col_name: str) -> pl.LazyFrame:
    """
    Apply ASCII decoder to a column using Polars map_elements.
    Falls back gracefully if column doesn't exist.
    """
    schema = df.collect_schema()
    if col_name not in schema.names():
        print(f"   ⚠️ Column '{col_name}' not found, skipping")
        return df
    
    print(f"   🔓 Decoding '{col_name}'...")
    
    return df.with_columns(
        pl.col(col_name)
        .cast(pl.Utf8)
        .map_elements(decode_ascii, return_dtype=pl.Utf8)
        .alias(col_name)
    )


print("✅ Decoder functions defined")

## Step 2: Load & Select Core Columns (Lazy)

In [ ]:
print("="*70)
print("📂 STEP 2: LOADING & COLUMN SELECTION")
print("="*70)

# Scan parquet (lazy - no memory load yet)
print("\n📂 Scanning parquet...")
q = pl.scan_parquet(RAW_TRAIN)

# Check available columns
available = q.collect_schema().names()
print(f"   Available columns: {len(available)}")

# Select only core columns that exist
keep_cols = [c for c in CORE_COLUMNS if c in available]
missing = [c for c in CORE_COLUMNS if c not in available]

print(f"   ✅ Keeping: {keep_cols}")
if missing:
    print(f"   ⚠️ Missing: {missing}")

q = q.select(keep_cols)
print("\n✅ Core columns selected (lazy)")

## Step 3: Decode String Columns

In [ ]:
print("="*70)
print("🔓 STEP 3: DECODING STRING COLUMNS")
print("="*70)

# Columns to decode
DECODE_COLS = ['id', 'gender', 'insulin_type_bolus']

# Note: map_elements requires eager evaluation
# We'll collect a sample first to verify, then process in chunks

print("\n📋 Collecting data for decoding...")
print("   (This may take a few minutes for large datasets)")

# Collect the lazy frame
df = q.collect()
print(f"   Shape: {df.shape}")

# Apply decoder to each column
for col in DECODE_COLS:
    if col in df.columns:
        print(f"   🔓 Decoding '{col}'...")
        df = df.with_columns(
            pl.col(col)
            .cast(pl.Utf8)
            .map_elements(decode_ascii, return_dtype=pl.Utf8)
            .alias(col)
        )

# Verify decoding
print("\n📋 Sample of decoded columns:")
for col in DECODE_COLS:
    if col in df.columns:
        unique = df.select(col).unique().head(5).to_series().to_list()
        print(f"   {col}: {unique}")

print("\n✅ Decoding complete!")

## Step 4: Data Cleaning & Preparation

In [ ]:
print("="*70)
print("🧹 STEP 4: DATA CLEANING")
print("="*70)

original_count = len(df)

# 1. Filter out null/empty IDs
print("\n1️⃣ Filtering null/empty IDs...")
df = df.filter(
    pl.col('id').is_not_null() & 
    (pl.col('id') != '') & 
    (pl.col('id') != 'Unknown')
)
print(f"   Rows after ID filter: {len(df):,}")

# 2. Convert date (assume Unix timestamp if numeric)
print("\n2️⃣ Converting date column...")
if 'date' in df.columns:
    dtype = df.schema['date']
    print(f"   Date dtype: {dtype}")
    
    if dtype in [pl.Int64, pl.Float64]:
        # Unix timestamp (seconds or milliseconds)
        df = df.with_columns(
            pl.from_epoch(pl.col('date'), time_unit='s').alias('date')
        )
    elif dtype == pl.Utf8:
        df = df.with_columns(
            pl.col('date').str.to_datetime().alias('date')
        )
    print(f"   Converted to: {df.schema['date']}")

# 3. Fill NaNs in numeric columns
print("\n3️⃣ Filling null values...")
df = df.with_columns([
    pl.col('carbs').fill_null(0.0).cast(pl.Float32),
    pl.col('basal').fill_null(0.0).cast(pl.Float32),
    pl.col('bolus').fill_null(0.0).cast(pl.Float32),
    pl.col('CGM').cast(pl.Float32),
    pl.col('age').fill_null(-1.0).cast(pl.Float32),
    pl.col('weight').fill_null(-1.0).cast(pl.Float32),
])

# 4. Filter null CGM
print("\n4️⃣ Filtering null CGM...")
df = df.filter(pl.col('CGM').is_not_null())
print(f"   Rows after CGM filter: {len(df):,}")

# 5. Sort by ID and Date (CRITICAL for delta calculation)
print("\n5️⃣ Sorting by [id, date]...")
df = df.sort(['id', 'date'])

dropped = original_count - len(df)
print(f"\n📊 Cleaning Summary:")
print(f"   Original: {original_count:,}")
print(f"   Final:    {len(df):,}")
print(f"   Dropped:  {dropped:,} ({dropped/original_count*100:.2f}%)")

## Step 5: Advanced Feature Engineering

In [ ]:
print("="*70)
print("🧬 STEP 5: ADVANCED FEATURE ENGINEERING")
print("="*70)

# ============================================================
# CGM TREND ANALYSIS (Deltas)
# ============================================================
print("\n📈 Creating CGM Deltas...")

df = df.with_columns([
    # CGM deltas (using window over patient ID)
    (pl.col('CGM') - pl.col('CGM').shift(1).over('id')).alias('cgm_delta_5min'),
    (pl.col('CGM') - pl.col('CGM').shift(3).over('id')).alias('cgm_delta_15min'),
    (pl.col('CGM') - pl.col('CGM').shift(6).over('id')).alias('cgm_delta_30min'),
    
    # Basal delta (to detect auto-mode corrections)
    (pl.col('basal') - pl.col('basal').shift(1).over('id')).alias('basal_delta_5min'),
])

# CGM Velocity (rate of change per minute)
df = df.with_columns(
    (pl.col('cgm_delta_15min') / 15).alias('cgm_velocity')
)

print("   ✅ cgm_delta_5min, cgm_delta_15min, cgm_delta_30min")
print("   ✅ cgm_velocity, basal_delta_5min")

In [ ]:
# ============================================================
# CLINICAL PARAMETERS: TDD, ISF, ICR
# ============================================================
print("\n💊 Calculating Clinical Parameters...")

# Calculate TDD (Total Daily Dose) per patient-day
# TDD = (basal_rate * 24) + sum(bolus)
# We'll use a rolling window approach for a moving TDD estimate

# For simplicity, calculate 24-hour rolling sum of insulin
# 24 hours = 288 samples at 5-min intervals
SAMPLES_24H = 288

df = df.with_columns([
    # Rolling bolus sum (24h)
    pl.col('bolus').rolling_sum(window_size=SAMPLES_24H).over('id').alias('bolus_24h'),
    # Average basal rate (24h)
    pl.col('basal').rolling_mean(window_size=SAMPLES_24H).over('id').alias('avg_basal_24h'),
])

# TDD estimation
df = df.with_columns(
    (pl.col('avg_basal_24h') * 24 + pl.col('bolus_24h')).alias('est_tdd')
)

# ISF (1800 rule) and ICR (500 rule)
df = df.with_columns([
    (1800 / pl.col('est_tdd').clip(lower_bound=1)).clip(upper_bound=200).alias('est_isf'),
    (500 / pl.col('est_tdd').clip(lower_bound=1)).clip(upper_bound=50).alias('est_icr'),
])

print("   ✅ est_tdd, est_isf, est_icr")

In [ ]:
# ============================================================
# IOB (INSULIN ON BOARD) CALCULATION
# ============================================================
print("\n💉 Calculating IOB (Insulin On Board)...")

# IOB uses 4-hour linear decay for rapid-acting insulin
# At 5-min intervals, 4 hours = 48 samples
# Decay factor per sample: (remaining_time / total_time)

IOB_DURATION_SAMPLES = 48  # 4 hours at 5-min intervals

# Create weighted sum for IOB using exponential decay approximation
# Simplified: Use rolling sum with decreasing weights
# For a linear decay model: IOB = sum(bolus_i * (1 - i/48)) for i in 0..47

# Simpler approach: Rolling sum with 50% weight for older insulin
df = df.with_columns([
    # Recent IOB (last 2 hours, full activity)
    pl.col('bolus').rolling_sum(window_size=24).over('id').fill_null(0).alias('iob_2h'),
    # Decaying IOB (2-4 hours, 50% activity)
    (pl.col('bolus').shift(24).over('id').fill_null(0)
     .rolling_sum(window_size=24).over('id').fill_null(0) * 0.5).alias('iob_decay'),
])

df = df.with_columns(
    (pl.col('iob_2h') + pl.col('iob_decay')).alias('iob')
)

# Clean up intermediate columns
df = df.drop(['iob_2h', 'iob_decay'])

print("   ✅ iob (4-hour decay model)")

In [ ]:
# ============================================================
# TIME FEATURES
# ============================================================
print("\n⏰ Creating Time Features...")

df = df.with_columns([
    pl.col('date').dt.hour().alias('hour'),
    pl.col('date').dt.weekday().alias('weekday'),
])

df = df.with_columns([
    (2 * np.pi * pl.col('hour') / 24).sin().cast(pl.Float32).alias('time_sin'),
    (2 * np.pi * pl.col('hour') / 24).cos().cast(pl.Float32).alias('time_cos'),
    pl.when(pl.col('weekday') >= 5).then(1.0).otherwise(0.0).cast(pl.Float32).alias('is_weekend'),
])

print("   ✅ time_sin, time_cos, is_weekend")

In [ ]:
# ============================================================
# GENDER ENCODING
# ============================================================
print("\n🏷️ Encoding Gender...")

df = df.with_columns(
    pl.when(pl.col('gender').str.to_lowercase() == 'male').then(1.0)
    .when(pl.col('gender').str.to_lowercase() == 'female').then(0.0)
    .otherwise(-1.0).cast(pl.Float32).alias('gender_encoded')
)

print("   ✅ gender_encoded (Male=1, Female=0, Unknown=-1)")

## Step 6: Final Cleanup & Output

In [ ]:
print("="*70)
print("📋 STEP 6: FINAL CLEANUP")
print("="*70)

# Drop intermediate columns
drop_cols = ['bolus_24h', 'avg_basal_24h', 'hour', 'weekday']
for col in drop_cols:
    if col in df.columns:
        df = df.drop(col)

# Cast all numeric columns to Float32 for efficiency
numeric_cols = ['cgm_delta_5min', 'cgm_delta_15min', 'cgm_delta_30min',
                'cgm_velocity', 'basal_delta_5min', 'est_tdd', 'est_isf', 
                'est_icr', 'iob']

for col in numeric_cols:
    if col in df.columns:
        df = df.with_columns(pl.col(col).cast(pl.Float32))

# Drop rows with null features (from shift operations)
before = len(df)
df = df.drop_nulls(subset=['cgm_delta_5min', 'cgm_delta_15min'])
after = len(df)
print(f"   Dropped {before - after:,} null rows from shift operations")

# Final schema
print(f"\n📊 Final Schema ({len(df.columns)} columns):")
for col, dtype in df.schema.items():
    print(f"   {col}: {dtype}")

print(f"\n📊 Final Shape: {df.shape}")

In [ ]:
# ============================================================
# SAVE OUTPUT
# ============================================================
print("="*70)
print("💾 SAVING OUTPUT")
print("="*70)

print(f"\n💾 Writing to: {OUTPUT_FILE}")
df.write_parquet(OUTPUT_FILE, compression='zstd')

# Verify
if os.path.exists(OUTPUT_FILE):
    size_gb = os.path.getsize(OUTPUT_FILE) / (1024**3)
    print(f"\n✅ File saved successfully!")
    print(f"   Size: {size_gb:.2f} GB")
    print(f"   Rows: {len(df):,}")
    print(f"   Columns: {len(df.columns)}")

# Show sample
print("\n📋 Sample Data:")
df.head(5)

In [ ]:
# ============================================================
# VERIFICATION
# ============================================================
print("="*70)
print("✅ VERIFICATION")
print("="*70)

# Check ID decoding
print("\n🔍 Sample Patient IDs:")
print(df.select('id').unique().head(10))

# Check gender decoding
print("\n🔍 Gender Distribution:")
print(df.group_by('gender').agg(pl.len().alias('count')).sort('count', descending=True))

# Check feature statistics
print("\n🔍 Feature Statistics:")
print(df.select(['CGM', 'cgm_delta_5min', 'cgm_velocity', 'iob', 'est_tdd']).describe())

print("\n🎉 RECONSTRUCTION COMPLETE!")